## 1. Install & import

In [1]:
!pip install -q open3d plotly

In [2]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

print("Open3D version:", o3d.__version__)

Open3D version: 0.20.0


## 2. Helper functions for Colab visualisation

In [3]:
def show_points(pcd, title, max_pts=6000, normals=False, n_normals=400, length=0.08):
    pts = np.asarray(pcd.points)
    if len(pts) > max_pts:
        pts = pts[np.random.choice(len(pts), max_pts, replace=False)]
    fig = go.Figure(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode="markers",
                                 marker=dict(size=1.8, color=pts[:,2], colorscale="Viridis")))
    if normals:
        P = np.asarray(pcd.points); N = np.asarray(pcd.normals)
        idx = np.random.choice(len(P), min(n_normals, len(P)), replace=False)
        xs, ys, zs = [], [], []
        for p, n in zip(P[idx], N[idx]):
            q = p + length * n
            xs += [p[0], q[0], None]; ys += [p[1], q[1], None]; zs += [p[2], q[2], None]
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="red", width=3)))
    fig.update_layout(title=title, scene=dict(aspectmode="data"), height=600, showlegend=False)
    fig.show()

def show_mesh(mesh, title, max_tris=150000):
    m = o3d.geometry.TriangleMesh(mesh)           # copy, so the original is untouched
    if len(m.triangles) > max_tris:               # simplify only for display
        m = m.simplify_quadric_decimation(max_tris)
    V = np.asarray(m.vertices); F = np.asarray(m.triangles)
    fig = go.Figure(go.Mesh3d(x=V[:,0], y=V[:,1], z=V[:,2], i=F[:,0], j=F[:,1], k=F[:,2],
                              intensity=V[:,2], colorscale="Viridis", flatshading=False))
    fig.update_layout(title=title, scene=dict(aspectmode="data"), height=600)
    fig.show()

## 3. Generate a synthetic noisy sphere point cloud

In [4]:
np.random.seed(42)
n = 5000
phi = np.random.uniform(0, 2 * np.pi, n)
theta = np.random.uniform(0, np.pi, n)
radius = 1.0

x = radius * np.sin(theta) * np.cos(phi)
y = radius * np.sin(theta) * np.sin(phi)
z = radius * np.cos(theta)
points = np.column_stack((x, y, z))

# Add small measurement noise
noise = np.random.normal(0, 0.015, points.shape)
points_noisy = points + noise

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_noisy)
print("Number of points:", len(pcd.points))

# To use a real point cloud instead, uncomment:
# pcd = o3d.io.read_point_cloud("input_point_cloud.ply")

Number of points: 5000


## 4. Visualise the input point cloud

In [5]:
show_points(pcd, "Input Point Cloud")

## 5. Estimate and orient surface normals

In [6]:
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.15, max_nn=30)
)
# Make neighbouring normals more consistent
pcd.orient_normals_consistent_tangent_plane(50)
print("Surface normals estimated.")

Surface normals estimated.


In [7]:
show_points(pcd, "Point Cloud with Normals (red lines, 400 sampled points)", normals=True)

## 6. Poisson Surface Reconstruction

In [8]:
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=9, width=0, scale=1.1, linear_fit=False
)
print("Poisson reconstruction completed.")
print("Initial mesh vertices :", len(mesh.vertices))
print("Initial mesh triangles:", len(mesh.triangles))

Poisson reconstruction completed.
Initial mesh vertices : 16740
Initial mesh triangles: 33300


In [9]:
show_mesh(mesh, "Poisson Mesh (before density filtering)")

## 7. Remove weakly supported (low-density) regions

In [10]:
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.02)
vertices_to_remove = densities < density_threshold
mesh.remove_vertices_by_mask(vertices_to_remove)

print("After density filtering:")
print("Vertices :", len(mesh.vertices))
print("Triangles:", len(mesh.triangles))

mesh.compute_vertex_normals()

After density filtering:
Vertices : 16405
Triangles: 32097


TriangleMesh with 16405 points and 32097 triangles.

In [11]:
show_mesh(mesh, "Poisson Reconstructed Mesh (filtered)")

## 8. Save the mesh and download it

In [12]:
output_file = "poisson_reconstructed_mesh.ply"
o3d.io.write_triangle_mesh(output_file, mesh)
print("Mesh saved as:", output_file)

try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    pass

Mesh saved as: poisson_reconstructed_mesh.ply


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Student tasks (parameter experiments)
Each run below uses the same noisy point cloud. Note the vertex / triangle counts for your report.

In [13]:
# Task 3: effect of Poisson depth (depth 10 needs more RAM; if Colab crashes, use 7, 8, 9)
results = {}
for d in [7, 9, 10]:
    m, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=d, width=0, scale=1.1, linear_fit=False)
    results[d] = m
    print(f"depth={d:2d} -> vertices={len(m.vertices):8d}, triangles={len(m.triangles):8d}")

depth= 7 -> vertices=   15526, triangles=   30936
depth= 9 -> vertices=   16740, triangles=   33300
depth=10 -> vertices=   16739, triangles=   33298


In [14]:
for d, m in results.items():
    show_mesh(m, f"Poisson mesh, depth = {d}", max_tris=100000)

In [15]:
# Task 4: effect of the normal-estimation radius (angle error vs. the true radial direction of the sphere)
for r in [0.05, 0.15, 0.40]:
    p = o3d.geometry.PointCloud(pcd)
    p.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=r, max_nn=30))
    p.orient_normals_consistent_tangent_plane(50)
    N = np.asarray(p.normals); P = np.asarray(p.points)
    true_n = P / np.linalg.norm(P, axis=1, keepdims=True)
    cos = np.abs(np.sum(N * true_n, axis=1)).clip(0, 1)      # abs(): ignores overall flip
    print(f"radius={r:.2f} -> mean normal error = {np.degrees(np.arccos(cos)).mean():.2f} degrees")

radius=0.05 -> mean normal error = 45.18 degrees
radius=0.15 -> mean normal error = 4.58 degrees
radius=0.40 -> mean normal error = 3.88 degrees


In [16]:
# Task 5: density threshold 2% vs 5%
for q in [0.02, 0.05]:
    m, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9, width=0, scale=1.1, linear_fit=False)
    dens = np.asarray(dens)
    m.remove_vertices_by_mask(dens < np.quantile(dens, q))
    print(f"threshold={int(q*100)}% -> vertices={len(m.vertices)}, triangles={len(m.triangles)}")

threshold=2% -> vertices=16405, triangles=32097
threshold=5% -> vertices=15903, triangles=30487
